# if 분기와 모델 선택 비교

같은 질문·자료·모델을 사용할 때 코드가 정한 조회와 모델이 고른 조회는 토큰, 지연, 실행 경로에서 어떻게 다를까요?

## 실행 환경 준비

Python 3.12 새 커널에서 첫 셀부터 순서대로 실행합니다. API 키 값은 출력하지 않습니다.

## 패키지와 환경 준비

고정한 패키지 버전과 API 키 존재 여부만 확인합니다. 키 값 자체는 출력하지 않습니다.

## 코드를 읽기 전에

두 경로는 같은 질문·도구·최종 답 형식을 사용하고 분기 주체만 바꿉니다. `fixed` 경로는 코드의 조건문이 규정 조회 도구 사용 여부를 정하고, `agent` 경로는 도구가 바인딩된 모델의 `tool_calls`를 읽어 실행 여부를 정합니다. `run_branch()`는 한 실행의 경로, 토큰, 벽시계 시간, 답을 사전으로 반환합니다.

각 질문을 두 경로에서 두 번 실행해 문자열 변동과 평균을 계산합니다. 표의 차이는 이 질문 집합과 실행 시점에서 관찰한 값입니다. 모델 경로의 호출 수가 늘었다는 사실만으로 답의 품질이 높아졌다고 말할 수 없습니다.

In [1]:
# 패키지와 환경 준비에 필요한 연산을 실행합니다.
%pip install -q langgraph==1.2.11 langchain==1.4.0 langchain-openai==1.6.0 python-dotenv==1.2.3
import os
from pathlib import Path
from importlib.metadata import version
from dotenv import load_dotenv

for folder in [Path.cwd(), *Path.cwd().parents]:
    if (folder / '.env').is_file():
        load_dotenv(folder / '.env')
        break
assert os.getenv('OPENAI_API_KEY'), 'OPENAI_API_KEY가 필요합니다.'
print('langgraph', version('langgraph'))
print('langchain', version('langchain'))
print('langchain-openai', version('langchain-openai'))
print('API 키 존재 여부 확인 완료')

Note: you may need to restart the kernel to use updated packages.
langgraph 1.2.11
langchain 1.4.0
langchain-openai 1.6.0
API 키 존재 여부 확인 완료


## 모델과 비용 함수 준비

실습에 사용할 모델과 토큰 사용량으로 추정 비용을 계산하는 함수를 준비합니다.

In [2]:
# 모델과 비용 함수 준비에 필요한 연산을 실행합니다.
from time import perf_counter
from typing import Any
from langchain.messages import AIMessage
from langchain_core.language_models import BaseChatModel
from langchain.chat_models import init_chat_model

MODEL_NAME = 'gpt-5.4-mini'
INPUT_USD_PER_MTOK = 0.75
OUTPUT_USD_PER_MTOK = 4.50

def usage_of(message: AIMessage) -> dict:
    usage = message.usage_metadata
    assert usage is not None, 'usage_metadata가 없어 비용을 계산할 수 없습니다.'
    assert 'input_tokens' in usage and 'output_tokens' in usage
    input_tokens = usage['input_tokens']
    output_tokens = usage['output_tokens']
    cost = input_tokens / 1_000_000 * INPUT_USD_PER_MTOK
    cost += output_tokens / 1_000_000 * OUTPUT_USD_PER_MTOK
    return {'input_tokens': input_tokens, 'output_tokens': output_tokens, 'estimated_cost_usd': cost}

def make_model() -> BaseChatModel:
    return init_chat_model(f'openai:{MODEL_NAME}', reasoning_effort='none')

## 읽기 전용 규정 도구 준비

두 경로가 함께 사용할 대출·운영·연체 규정 조회 도구를 정의합니다.

In [3]:
# 읽기 전용 규정 도구 준비에 필요한 연산을 실행합니다.
from langchain.tools import tool

POLICIES = {
    '대출': '일반 도서는 1인 5권까지 14일 동안 대출할 수 있습니다.',
    '운영': '평일은 09:00~20:00, 토요일은 10:00~17:00에 운영합니다.',
    '연체': '연체한 일수만큼 새 대출이 제한됩니다.',
}

@tool
def lookup_policy(topic: str) -> str:
    """교육센터 자료실 규정을 주제어로 조회합니다. topic은 대출, 운영, 연체 중 하나입니다."""
    if topic not in POLICIES:
        raise ValueError(f'지원하지 않는 주제: {topic}')
    return POLICIES[topic]

TOOLS = [lookup_policy]

## 공통 답변 프롬프트와 측정 함수

두 경로가 같은 답변 조건을 쓰도록 공통 프롬프트와 측정 함수를 준비합니다.

In [4]:
# 공통 답변 프롬프트와 측정 함수에 필요한 연산을 실행합니다.
from langchain.messages import HumanMessage, ToolMessage

QUESTIONS = [
    '일반 도서는 며칠 동안 빌릴 수 있나요?',
    '토요일 자료실 운영 시간을 알려 주세요.',
    '책을 사흘 연체하면 새 대출은 어떻게 되나요?',
]

def timed_invoke(model: Any, messages: list) -> tuple[AIMessage, float]:
    started = perf_counter()
    reply = model.invoke(messages)
    return reply, perf_counter() - started

## 고정 주제 규칙

질문의 키워드를 대출·운영·연체 주제로 연결하는 고정 규칙을 정의합니다.

In [5]:
# 고정 주제 규칙에 필요한 연산을 실행합니다.
def fixed_topic(question: str) -> str:
    if '연체' in question:
        return '연체'
    if '운영' in question or '시간' in question:
        return '운영'
    return '대출'

## 공통 그래프 상태

질문, 전략, 메시지, trace와 사용량을 담는 공통 그래프 상태를 정의합니다.

In [6]:
# 공통 그래프 상태에 필요한 연산을 실행합니다.
from typing import Annotated, Literal
from typing_extensions import TypedDict
from langchain.messages import AIMessage, AnyMessage
from langgraph.graph.message import add_messages

class CompareState(TypedDict):
    messages: Annotated[list[AnyMessage], add_messages]
    trace: list[str]
    measurements: list[dict]
    strategy: Literal['fixed', 'agent']

## 고정 또는 모델 선택 노드

fixed는 규칙으로, agent는 모델로 같은 형식의 도구 호출 AIMessage를 만듭니다.

In [7]:
# 고정 또는 모델 선택 노드에 필요한 연산을 실행합니다.
choice_model = make_model().bind_tools(TOOLS)

def choose(state: CompareState) -> dict:
    if state['strategy'] == 'fixed':
        question = state['messages'][0].text
        call = {'name': 'lookup_policy', 'args': {'topic': fixed_topic(question)},
                'id': 'fixed_call', 'type': 'tool_call'}
        return {'messages': [AIMessage(content='', tool_calls=[call])],
                'trace': state['trace'] + ['fixed_choose']}
    reply, latency = timed_invoke(choice_model, state['messages'])
    return {'messages': [reply], 'trace': state['trace'] + ['model_choose'],
            'measurements': state['measurements'] + [{**usage_of(reply), 'latency_sec': latency}]}

## 공통 도구·답변 노드

두 경로가 공유할 ToolNode 표시 단계와 최종 답변 모델 노드를 정의합니다.

In [8]:
# 공통 도구·답변 노드에 필요한 연산을 실행합니다.
answer_model = make_model()

def mark_tools(state: CompareState) -> dict:
    return {'trace': state['trace'] + ['tools']}

def answer(state: CompareState) -> dict:
    reply, latency = timed_invoke(answer_model, state['messages'])
    return {'messages': [reply], 'trace': state['trace'] + ['model_answer'],
            'measurements': state['measurements'] + [{**usage_of(reply), 'latency_sec': latency}]}

## 같은 ToolNode 그래프 조립

choose 방식만 다르게 연결한 뒤 공유 노드를 하나의 StateGraph에 조립합니다.

In [9]:
# 같은 ToolNode 그래프 조립에 필요한 연산을 실행합니다.
from langgraph.graph import StateGraph, START
from langgraph.prebuilt import ToolNode, tools_condition

builder = StateGraph(CompareState)
builder.add_node('choose', choose)
builder.add_node('mark_tools', mark_tools)
builder.add_node('tools', ToolNode(TOOLS))
builder.add_node('answer', answer)
builder.add_edge(START, 'choose')
builder.add_conditional_edges('choose', tools_condition, {'tools': 'mark_tools', '__end__': '__end__'})
builder.add_edge('mark_tools', 'tools')
builder.add_edge('tools', 'answer')
graph = builder.compile()

## 공통 그래프 실행 함수

전략과 질문을 받아 그래프 전체 지연과 도구 메시지 존재 여부를 기록합니다.

In [10]:
# 공통 그래프 실행 함수에 필요한 연산을 실행합니다.
def run_branch(question: str, strategy: str) -> dict:
    prompt = f'{question} 반드시 제공된 규정 조회 도구로 확인한 뒤 한 문장으로 답하세요.'
    started = perf_counter()
    state = graph.invoke({'messages': [HumanMessage(prompt)], 'trace': [],
                          'measurements': [], 'strategy': strategy})
    latency = perf_counter() - started
    assert any(type(message).__name__ == 'ToolMessage' for message in state['messages'])
    measures = state['measurements']
    total_input = sum(m['input_tokens'] for m in measures)
    total_output = sum(m['output_tokens'] for m in measures)
    total_cost = sum(m['estimated_cost_usd'] for m in measures)
    return {'branch': strategy, 'question': question, 'trace': state['trace'],
            'answer': state['messages'][-1].text, 'latency_sec': latency,
            'input_tokens': total_input, 'output_tokens': total_output,
            'estimated_cost_usd': total_cost}

## 같은 조건으로 두 번씩 실제 실행

세 질문을 fixed와 agent 경로에서 각각 두 번 실행해 비교 자료를 수집합니다.

In [11]:
# 같은 조건으로 두 번씩 실제 실행에 필요한 연산을 실행합니다.
records = []
for repeat in range(1, 3):
    for question in QUESTIONS:
        for strategy in ('fixed', 'agent'):
            record = run_branch(question, strategy)
            record['repeat'] = repeat
            records.append(record)
            print(repeat, record['branch'], record['trace'], round(record['estimated_cost_usd'], 6))

total_cost = sum(r['estimated_cost_usd'] for r in records)
print('chapter_section_estimated_cost_usd:', round(total_cost, 6))
assert total_cost < 1.0

1 fixed ['fixed_choose', 'tools', 'model_answer'] 0.000175


1 agent ['model_choose', 'tools', 'model_answer'] 0.00039


1 fixed ['fixed_choose', 'tools', 'model_answer'] 0.000158


1 agent ['model_choose', 'tools', 'model_answer'] 0.000367


1 fixed ['fixed_choose', 'tools', 'model_answer'] 0.000162


...

2 agent ['model_choose', 'tools', 'model_answer'] 0.00039


2 fixed ['fixed_choose', 'tools', 'model_answer'] 0.000158


2 agent ['model_choose', 'tools', 'model_answer'] 0.000367


2 fixed ['fixed_choose', 'tools', 'model_answer'] 0.000176


2 agent ['model_choose', 'tools', 'model_answer'] 0.000409
chapter_section_estimated_cost_usd: 0.003314


## 토큰·지연·출력 변동 집계

경로별 토큰·추정 비용·지연과 서로 다른 답변 문자열의 개수를 집계합니다.

In [12]:
# 토큰·지연·출력 변동 집계에 필요한 연산을 실행합니다.
from collections import defaultdict
from statistics import mean

groups = defaultdict(list)
for record in records:
    groups[record['branch']].append(record)

for branch, rows in groups.items():
    variants = {q: len({r['answer'] for r in rows if r['question'] == q}) for q in QUESTIONS}
    print({
        'branch': branch,
        'runs': len(rows),
        'mean_input_tokens': round(mean(r['input_tokens'] for r in rows), 1),
        'mean_output_tokens': round(mean(r['output_tokens'] for r in rows), 1),
        'mean_latency_sec': round(mean(r['latency_sec'] for r in rows), 3),
        'total_estimated_cost_usd': round(sum(r['estimated_cost_usd'] for r in rows), 6),
        'output_variants_by_question': variants,
    })
print('sample_traces:', {b: groups[b][0]['trace'] for b in groups})

{'branch': 'fixed', 'runs': 6, 'mean_input_tokens': 88, 'mean_output_tokens': 22.5, 'mean_latency_sec': 1.077, 'total_estimated_cost_usd': 0.001004, 'output_variants_by_question': {'일반 도서는 며칠 동안 빌릴 수 있나요?': 2, '토요일 자료실 운영 시간을 알려 주세요.': 1, '책을 사흘 연체하면 새 대출은 어떻게 되나요?': 2}}
{'branch': 'agent', 'runs': 6, 'mean_input_tokens': 264.3, 'mean_output_tokens': 41.5, 'mean_latency_sec': 1.462, 'total_estimated_cost_usd': 0.00231, 'output_variants_by_question': {'일반 도서는 며칠 동안 빌릴 수 있나요?': 2, '토요일 자료실 운영 시간을 알려 주세요.': 1, '책을 사흘 연체하면 새 대출은 어떻게 되나요?': 2}}
sample_traces: {'fixed': ['fixed_choose', 'tools', 'model_answer'], 'agent': ['model_choose', 'tools', 'model_answer']}


## 다른 상황에 적용하기

‘환불 규정’처럼 키워드와 처리 경로가 안정적인 요청은 고정 분기가 예측 가능할 수 있습니다. 같은 문장에 배송·회원 등 여러 의도가 섞이면 모델 선택이 더 유연할 수 있지만 잘못된 도구 선택도 측정해야 합니다. 비교할 때는 정답률, 불필요한 호출, 미호출, 지연, 출력 변동을 같은 평가셋에서 함께 기록합니다.